In [3]:
import pandas as pd
import re

## Extracción del archivo

In [4]:
ruta_archivo = "../data/unprocessed_files/SABI_Export_2.txt"
df = pd.read_csv(
    ruta_archivo, sep=";", index_col=0, encoding="utf-16",
    dtype={
        'Código primario CNAE 2009': str,
        'Código(s) segundario(s) CNAE 2009': str
    }
)
#, sep=";"  encoding="utf-16"
columns = list(df.columns)
columns_sql = ', '.join(columns)
placeholders = ", ".join(f":{c}" for c in columns)
print(columns)
print(columns_sql)
print(placeholders)


FileNotFoundError: [Errno 2] No such file or directory: '../data/unprocessed_files/SABI_Export_2.txt'

## Modificación del archivo

In [ ]:

# 1. Eliminamos columnas innecesarias para el estudio
# df = df.drop(columns=['Código consolidación', 'País'])

# 2. Cambiamos al formato fecha solo indicando el año
df['Ultimo año disponible'] = pd.to_datetime(df['Ultimo año disponible'], format='%d/%m/%Y', errors='coerce').dt.year
df['Fecha constitución'] = pd.to_datetime(df['Fecha constitución'], format='%d/%m/%Y', errors='coerce').dt.year

# 3. Renombre de todas las columnas
df = df.rename(columns={
    'Nombre': 'Company',
    'Código NIF': 'NIF_cod',
    'Calle' : 'Street',
    'Código postal' : 'ZIP_cod',
    'Localidad': 'City',
    'Provincia' : 'Province',
    'Código primario CNAE 2009' : 'CNAE_primary',
    'Código(s) segundario(s) CNAE 2009' : 'CNAE_secundary',
    'Número empleados Últ. año disp.' : 'N_emp',
    'Fecha constitución' : 'Date_Inc',
    'Ultimo año disponible' : 'Last_year',
    'Ingresos de explotación mil EUR Últ. año disp.' : 'Earnings_mil_last_year',
    'Coordenada - X' : "Longitude",
    'Coordenada - Y' : 'Latitude'
})

# 4. Limpiamos los valores nulos y adaptamos el texto a numerico
# df['Earnings_mil_last_year'] = df['Earnings_mil_last_year'].str.replace('.', '', regex=False)
df['Earnings_mil_last_year'] = pd.to_numeric(df['Earnings_mil_last_year'], errors='coerce')
df['N_emp'] = pd.to_numeric(df['N_emp'], errors='coerce')
not_geocoded = (df['Longitude'] == 0) & (df['Latitude'] == 0)
df.loc[not_geocoded, ['Longitude', 'Latitude']] = pd.NA

# 5. Limpieza de espacios y duplicados
columnas_texto = df.select_dtypes(include=['object','str']).columns
for col in columnas_texto:
    df[col] = df[col].astype(str).str.strip()

df = df.drop_duplicates(subset=['NIF_cod'])

In [ ]:
# 1. Cambia al formato fecha solo indicando el año
df['Last available year'] = pd.to_datetime(df['Last available year'], format='%d/%m/%Y', errors='coerce').dt.year
df['Date of Establishment'] = pd.to_datetime(df['Date of Establishment'], format='%d/%m/%Y', errors='coerce').dt.year

# 2. Renombre de todas las columnas
df = df.rename(columns={
    'NIF Code': 'NIF_Code',
    'CNAE 2009 Primary Code': 'CNAE_primary_Code',
    'CNAE 2009 Secondary Code(s)': 'CNAE_secundary_Code',
    'Number of employees Last avail. yr': 'N_emp',
    'Date of Establishment': 'Date_Est',
    'Last available year': 'Last_year',
    'Operating revenue / turnover th EUR Last avail. yr': 'Earnings_mil_last_year',
    'X coordinate' : 'Longitude',
    'Y coordinate' : 'Latitude'
})
df.columns = [c.replace(' ','_') for c in df.columns]

# 3. Limpieza de los valores nulos y adaptación del texto a numerico
df['Earnings_mil_last_year'] = pd.to_numeric(df['Earnings_mil_last_year'], errors='coerce')
df['N_emp'] = pd.to_numeric(df['N_emp'], errors='coerce')
not_geocoded = (df['Longitude'] == 0) & (df['Latitude'] == 0)
df.loc[not_geocoded, ['Longitude', 'Latitude']] = pd.NA

# 4. Limpieza de espacios y duplicados
text_columns = df.select_dtypes(include=['object', 'str']).columns
for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

df = df.drop_duplicates(subset=['NIF_Code'])

In [ ]:
print(df.columns.tolist())
#print(df['Código(s) segundario(s) CNAE 2009'].head(10))
faltantes = df['Latitude'].isna().sum()

print(f"Empresas sin coordenadas: {faltantes} de {len(df)} ({faltantes/len(df)*100:.1f}%)")

['Company', 'NIF_cod', 'Street', 'ZIP_cod', 'City', 'Province', 'CNAE_primary', 'CNAE_secundary', 'N_emp', 'Date_Inc', 'Last_year', 'Earnings_mil_last_year', 'Longitude', 'Latitude']
Empresas sin coordenadas: 16 de 14096 (0.1%)


## Exportación archivo limpio

In [ ]:
archivo_limpio = ruta_archivo.replace("ArchivosSinProcesar", "ArchivosProcesados").replace(".txt", "_limpio.csv")
df.to_csv(archivo_limpio, index=False, sep=";", encoding="utf-8-sig")

# ETL DATOS DE LA EMPRESA

In [ ]:
import pandas as pd

## Extracción de datos

In [ ]:
ruta_archivo = "../data/ArchivosSinProcesar/SABI_Export.xlsx"
df = pd.read_excel(ruta_archivo, index_col=0)
df.head(10)
print(df.columns[0])

Company Name


In [ ]:

metricas= [
    'Total activo',
    'Ingresos de explotación',
    'Importe neto Cifra de Ventas',
    'EBITDA',
    'EBIT',
    'Resultado del Ejercicio',
    'Ratio de liquidez',
    'Rentabilidad económica',
    'Rentabilidad financiera',
    'Endeudamiento'
    ]

df_final = pd.DataFrame()

for n in lista:
    columna = [col for col in df.columns if n in col]

    df_col = pd.melt(
    df,
    id_vars=['Código NIF'],          # La columna que se MANTIENE intacta por fila
    value_vars=columna,       # Las columnas que se van a convertir en filas (los años)
    var_name='Año',                   # Nombre de la nueva columna de años
    value_name=n     # Nombre de la nueva columna para los valores
    )

    df_col['Año'] = df_col['Año'].str.extract(r'(\d{4})').astype(int)

    df_final = pd.merge(
        df_final,
        df_col,
        on=['Código NIF', 'Año'],
        how='outer'   # 'outer' asegura que no se pierda ningún año/empresa aunque falte un dato en una de las dos
    )

    df_final = df_final.sort_values(
        by=['Código NIF', 'Año'],
        ascending=[True, False]
    ).reset_index(drop=True)

# 1. Identificamos las columnas de años (todas las que empiezan por "Total activo")
# Esto selecciona dinámicamente las columnas desde 2025 hasta 2001
columnas_activo = [col for col in df.columns if 'Total activo' in col]
columnas_ingresos = [col for col in df.columns if 'Ingresos de explotación' in col]
columnas_Ventas = [col for col in df.columns if 'Importe neto Cifra de Ventas' in col]
columnas_EBIT = [col for col in df.columns if 'EBIT' in col]
columnas_EBITDA = [col for col in df.columns if 'EBITDA' in col]
columnas_res_eje = [col for col in df.columns if 'Resultado del Ejercicio' in col]
columnas_marg_Ben = [col for col in df.columns if 'Margen de beneficio' in col]
columnas_liquidez = [col for col in df.columns if 'Ratio de liquidez' in col]
columnas_rent_econ = [col for col in df.columns if 'Rentabilidad económica' in col]
columnas_rent_fina = [col for col in df.columns if 'Rentabilidad financiera' in col]
columnas_endeudamiento = [col for col in df.columns if 'Endeudamiento' in col]

# 2. Aplicamos melt para despivotar la tabla
df_largo = pd.melt(
    df,
    id_vars=['Código NIF'],          # La columna que se MANTIENE intacta por fila
    value_vars=columnas_activo,       # Las columnas que se van a convertir en filas (los años)
    var_name='Año',                   # Nombre de la nueva columna de años
    value_name='Total_Activo_EUR'     # Nombre de la nueva columna para los valores
)

# 3. Limpiamos la columna 'Año' para quedarnos SOLO con el número del año (ej. '2025')
# Como SABI mete "Total activo\nmil EUR\n2025", extraemos los últimos 4 dígitos
df_largo['Año'] = df_largo['Año'].str.extract(r'(\d{4})').astype(int)

# 4. Ordenamos por Empresa (NIF) y luego por Año de forma descendente (de 2025 a 2001)
df_largo = df_largo.sort_values(by=['Código NIF', 'Año'], ascending=[True, False]).reset_index(drop=True)

print(df_largo.head(100))


   Código NIF   Año Total_Activo_EUR
0   A30106264  2025             n.d.
1   A30106264  2024       214067.165
2   A30106264  2023     221592.09064
3   A30106264  2022       211806.517
4   A30106264  2021       205784.355
..        ...   ...              ...
70  B30411318  2005        399.90745
71  B30411318  2004        293.88713
72  B30411318  2003        255.96951
73  B30411318  2002        201.38889
74  B30411318  2001        179.79995

[75 rows x 3 columns]


In [ ]:
metricas= [
    'Total activo',
    'Ingresos de explotación',
    'Importe neto Cifra de Ventas',
    'EBITDA',
    'EBIT',
    'Resultado del Ejercicio',
    'Ratio de liquidez',
    'Rentabilidad económica',
    'Rentabilidad financiera',
    'Endeudamiento'
    ]

patron_metricas = '|'.join(re.escape(m) for m in metricas)

# --- 3. Identificamos TODAS las columnas relevantes de una sola vez ---
columnas_metricas = [col for col in df.columns if re.search(patron_metricas, col)]

# --- 4. Un único melt para todas las métricas a la vez ---
df_largo = pd.melt(
    df,
    id_vars=['Código NIF'],
    value_vars=columnas_metricas,
    var_name='Columna_Original',
    value_name='Valor'
)

nombres_metricas = list(dict.fromkeys([
    c.split('\n')[0] for c in df.columns if len(c.split('\n')) >= 3 
]))
print(nombres_metricas)

['Total activo', 'Ingresos de explotación', 'Importe neto Cifra de Ventas', 'EBIT', 'EBITDA', 'Resultado del Ejercicio', 'Margen de beneficio (%)', 'Ratio de liquidez', 'Rentabilidad económica (%)', 'Rentabilidad financiera (%)', 'Endeudamiento (%)']


In [ ]:
'''metricas= [
    'Total activo',
    'Ingresos de explotación',
    'Importe neto Cifra de Ventas',
    'EBITDA',
    'EBIT',
    'Resultado del Ejercicio',
    'Ratio de liquidez',
    'Rentabilidad económica',
    'Rentabilidad financiera',
    'Endeudamiento'
    ]'''

# -- 1. Obtenemos todos los nombres de las variables financieras y de las variables financieras
#       que estan representadas en miles

metricas = list(dict.fromkeys([
    c.split('\n')[0] for c in df.columns if len(c.split('\n')) >= 3 
]))
print(nombres_metricas)

metricas = sorted(metricas, key=len, reverse=True)

metricas_en_miles = [
    m for m in metricas
    if any(('mil EUR' in col) and (m in col) for col in df.columns)
]


patron_metricas = '|'.join(re.escape(m) for m in metricas)

# --- 3. Identificamos TODAS las columnas relevantes de una sola vez ---
columnas_metricas = [col for col in df.columns if re.search(patron_metricas, col)]

# --- 4. Un único melt para todas las métricas a la vez ---
df_largo = pd.melt(
    df,
    id_vars=['Código NIF'],
    value_vars=columnas_metricas,
    var_name='Columna_Original',
    value_name='Valor'
)

# --- 5. Extraemos la métrica y el año en un solo paso con una regex de 2 grupos ---
extraido = df_largo['Columna_Original'].str.extract(
    rf'({patron_metricas}).*?(\d{{4}})',
    flags=re.DOTALL)
df_largo['Metrica'] = extraido[0]
df_largo['Año'] = extraido[1].astype(int)

# --- 6. Pivotamos: cada métrica pasa a ser su propia columna ---
df_final = df_largo.pivot_table(
    index=['Código NIF', 'Año'],
    columns='Metrica',
    values='Valor',
    aggfunc='first'   # 'first' porque no debería haber duplicados; evita errores si los hay
).reset_index()

df_final.columns.name = None  # Quita el nombre "Metrica" que pivot_table deja en el eje de columnas

# --- 7. Ordenamos una única vez, al final ---
df_final = df_final.sort_values(
    by=['Código NIF', 'Año'],
    ascending=[True, False]
).reset_index(drop=True)

columnas_valores = [c for c in df_final.columns if c not in ['Código NIF', 'Año']]
df_final[columnas_valores] = df_final[columnas_valores].apply(pd.to_numeric, errors='coerce')

for col in metricas_en_miles:
    df_final[col] = df_final[col] * 1000

df_final = df_final.dropna(subset=columnas_valores, how='all').reset_index(drop=True)

# --- Paso A: quitar cualquier sufijo entre paréntesis, ej. '(%)' ---
df_final.columns = [re.sub(r'\s*\([^)]*\)', '', c).strip() for c in df_final.columns]

# --- Paso B: sustituir espacios por guion bajo en todos los nombres de columna ---
df_final.columns = [c.replace(' ', '_') for c in df_final.columns]


['Total activo', 'Ingresos de explotación', 'Importe neto Cifra de Ventas', 'EBIT', 'EBITDA', 'Resultado del Ejercicio', 'Margen de beneficio (%)', 'Ratio de liquidez', 'Rentabilidad económica (%)', 'Rentabilidad financiera (%)', 'Endeudamiento (%)']


In [ ]:
print(df_final.head(10))
archivo_limpio = ruta_archivo.replace("ArchivosSinProcesar", "ArchivosProcesados").replace(".xlsx", "_limpio.csv")
df_final.to_csv(archivo_limpio, index=False, sep=";", encoding="utf-8-sig")

    NIF_Code  Year        EBIT      EBITDA  Economic_profitability  \
0  A30106264  2024  1842339.00  6227060.00                   0.863   
1  A30106264  2023  1492356.21  4772779.73                   0.283   
2  A30106264  2022 -1320547.00  1175210.00                  -0.054   
3  A30106264  2021  1125648.00  4032227.00                   0.532   
4  A30106264  2020   643478.00  4460040.00                   0.461   
5  A30106264  2019  4205939.00  8324911.00                   2.464   
6  A30106264  2018   558648.00  4597996.00                   0.236   
7  A30106264  2017  1493431.00  5204765.00                   0.771   
8  A30106264  2016  3624181.00  6821519.00                   4.956   
9  A30106264  2015  2456830.00  5899434.00                  -0.366   

   Financial_profitability  Indebtness  Liquidity_Ratio  \
0                    2.817      69.345            0.686   
1                    0.815      65.316            0.798   
2                   -0.137      60.455            0.

In [ ]:
# -- 2. Obtiene todos los nombres de las variables financieras y de las variables
#       que estan representadas en miles
metrics = list(dict.fromkeys([
    c.split('\n')[0] for c in df.columns if len(c.split('\n')) >= 3
]))

metrics = sorted(metrics, key=len, reverse=True)

metrics_in_thousands = [
    m for m in metrics
    if any(('th EUR' in col) and (m in col) for col in df.columns)
]

# Construye un patrón de búsqueda múltiple a partir de la lista de métricas.
metrics_pattern = '|'.join(re.escape(m) for m in metrics)

# -- 3. Identifica las columnas relevantes
metric_columns = [col for col in df.columns if re.search(metrics_pattern, col)]

# -- 4. Despivota los datos de métricas de la tabla (de columnas a filas): mantiene el NIF como
#        identificador fijo y transforma todas las columnas de métricas en una única columna
#        con su respectivo valor financiero.
df_melt = pd.melt(
    df,
    id_vars=['NIF Code'],
    value_vars=metric_columns,
    var_name='Original_Column',
    value_name='Value'
)

# -- 5. Extrae las metricas y el año de la columna unica y las separa en dos columnas
#       nuevas (Metric y Year)
extracted = df_melt['Original_Column'].str.extract(
    rf'({metrics_pattern}).*?(\d{{4}})',
    flags=re.DOTALL)
df_melt['Metric'] = extracted[0]
df_melt['Year'] = extracted[1].astype(int)

# -- 6. Pivota y cada métrica pasa a ser su propia columna, dejando las columnas de
#       NIF Code y Year
df_final = df_melt.pivot_table(
    index=['NIF Code', 'Year'],
    columns='Metric',
    values='Value',
    aggfunc='first'
).reset_index()

# Quita el nombre "Metric" que pivot_table deja en el eje de columnas
df_final.columns.name = None

# -- 7. Ordena las filas por NIF Code y Year
df_final = df_final.sort_values(
    by=['NIF Code', 'Year'],
    ascending=[True, False]
).reset_index(drop=True)

# Fuerza las columnas con metricas financieras a numero y las que no
# muestran valores como n.d. a NaN
value_columns = [c for c in df_final.columns if c not in ['NIF Code', 'Year']]
df_final[value_columns] = df_final[value_columns].apply(pd.to_numeric, errors='coerce')

# Multiplica por 1000 las métricas que venían en "th EUR" para dejarlas en el valor real
for col in metrics_in_thousands:
    df_final[col] = df_final[col] * 1000

# Elimina los años que no muestran ningun dato finacieros para la optimizacion del estudio
df_final = df_final.dropna(subset=value_columns, how='all').reset_index(drop=True)

# Quita cualquier sufijo entre paréntesis, ej. '(%)', sustituye barras y espacios por
# guion bajo en todos los nombres de las columnas
df_final.columns = [re.sub(r'\s*\([^)]*\)', '', c).strip() for c in df_final.columns]
df_final.columns = [c.replace('/', '_').replace(' ', '_') for c in df_final.columns]
df_final.columns = [re.sub(r'_+', '_', c).strip('_') for c in df_final.columns]

# DATOS GEOESPACIALES

In [ ]:
# import osmnx as ox
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt


# Carga el archivo (leerá automáticamente el .dbf y .prj de la misma carpeta)
secciones = gpd.read_file('España_Seccionado2026/SECC_CE_20260101.shp')
df = pd.read_csv('poblacion_por_censo_Madrid.csv', index_col=False, sep=';', encoding='utf-8', thousands='.')
df.columns = df.columns.str.strip()

print(df.columns)

df['Total'] = df['Total'].astype(float)
secciones = secciones[secciones['CUSEC'] == '2807901001'].copy()


# secciones = secciones[secciones['NMUN'] == 'Madrid'].copy()
df = df[
    (df['Secciones'] == '2807901001 Madrid sección 01001') & 
    (df['Sexo'] == 'Total') & 
    (df['Nacionalidad'] == 'Total')
].copy()

area_km2 = secciones.area.iloc[0] / 1000000

df['Densidad_hab_km2'] = df['Total'] / area_km2

print(secciones.area.iloc[0])
print(area_km2)

print(df[['Secciones', 'Periodo', 'Total', 'Densidad_hab_km2']])

'''
print(secciones.columns)
print(secciones['geometry'].head())
print(secciones.area)
'''
# Dibuja el mapa rápidamente
secciones.plot()

plt.show()

## FUNCIONES

In [ ]:
import pandas as pd

df = pd.read_excel('../data/unprocessed_geodata_files/renta_poblacional_españa.xlsx', thousands='.', skiprows=7, nrows=55730, usecols='A:F')

df.rename(columns={
    '' = 'secciones'
})


print(df.head(20))


/Users/joticasmiras/Desktop/TFM/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


                                                2023   2022   2021   2020  \
0                      01001 Alegría-Dulantzi  16429  15116  14647  13969   
1        0100101 Alegría-Dulantzi distrito 01  16429  15116  14647  13969   
2   0100101001 Alegría-Dulantzi sección 01001  17069  15473  15191  14553   
3   0100101002 Alegría-Dulantzi sección 01002  15869  14811  14172  13460   
4                               01002 Amurrio  17486  16011  15192  14683   
5                 0100201 Amurrio distrito 01  17486  16011  15192  14683   
6            0100201001 Amurrio sección 01001  17409  16168  15408  16061   
7            0100201002 Amurrio sección 01002  19517  17691  16964  15624   
8            0100201003 Amurrio sección 01003  17493  16572  15612  14549   
9            0100201004 Amurrio sección 01004  18738  16650  15441  15079   
10           0100201005 Amurrio sección 01005  15572  14271  14134  13770   
11           0100201006 Amurrio sección 01006  16946  15596  14544  13873   